# Lab 07 — Deploy the Multi-Agent System with Streamlit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-07-deploy-the-multi-agent-system-with-streamlit/lab-07-deploy-the-multi-agent-system-with-streamlit.ipynb)

**Topic:** 3 — Multi-Agent System Development with OpenAI Agents SDK

**Objective:** Deploy a collaborative multi-agent system as a shareable Streamlit web application

Put a web interface on the agent team from Lab 06 so a non-developer can use it, streaming the reply and showing which agent handled each request.

Full step-by-step instructions are in the Learner Guide.


> **Streamlit does not render inside Colab.** A Streamlit app is a server process driven by `streamlit run`, not a notebook widget — nothing will display in the output cell. This notebook therefore writes the app files to disk with `%%writefile`, and you run them **locally** with the command shown at the end. Treat the notebook as the place you assemble and read the code; treat your own machine as the place you run it.


In [ ]:
!pip install -q openai-agents pydantic python-dotenv streamlit


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("OPENAI_API_KEY")
print("Keys set:", [v for v in ["OPENAI_API_KEY"] if os.environ.get(v)])


## 1. Write the Lab 06 agent team to team.py

The app imports `triage_agent` from this module, so it must sit next to `app.py`.

**The filename must not be `agents.py`** — the SDK is imported as `from agents import ...`, so a local `agents.py` would shadow the installed package.


In [ ]:
%%writefile team.py
"""The Lab 06 agent team: three specialists behind a triage supervisor."""

from dotenv import load_dotenv
from pydantic import BaseModel

from agents import (
    Agent,
    GuardrailFunctionOutput,
    RunContextWrapper,
    Runner,
    function_tool,
    input_guardrail,
)

load_dotenv()

MODEL = "gpt-4o-mini"


@function_tool
def search_notes(query: str) -> str:
    """Search the internal knowledge base for background on a topic.

    Use this when answering a factual or research question.

    Args:
        query: The search terms.
    """
    notes = {
        "mrt": "The Thomson-East Coast Line opened in stages from 2020.",
        "python": "Python 3.11 introduced significant interpreter speedups.",
        "agents": "Multi-agent systems split work across specialised agents.",
    }
    hits = [text for key, text in notes.items() if key in query.lower()]
    return "\n".join(hits) if hits else "No relevant notes found."


@function_tool
def run_linter(code: str) -> str:
    """Check a Python snippet for syntax errors before returning it.

    Args:
        code: The Python source to check.
    """
    try:
        compile(code, "<submitted>", "exec")
    except SyntaxError as exc:
        return f"Syntax error on line {exc.lineno}: {exc.msg}"
    return "Syntax OK."


@function_tool
def count_words(text: str) -> str:
    """Count the words in a draft, to check it meets a length requirement.

    Args:
        text: The draft text.
    """
    return f"{len(text.split())} words."


research_agent = Agent(
    name="Research Agent",
    handoff_description="Answers factual and research questions using the knowledge base.",
    instructions=(
        "You are a research specialist. Answer factual questions using the "
        "search_notes tool. Cite what the notes said."
    ),
    model=MODEL,
    tools=[search_notes],
)

coding_agent = Agent(
    name="Coding Agent",
    handoff_description="Writes, explains and debugs Python code.",
    instructions=(
        "You are a Python specialist. Write clear, correct code with type hints. "
        "Always check your code with the run_linter tool before returning it."
    ),
    model=MODEL,
    tools=[run_linter],
)

writing_agent = Agent(
    name="Writing Agent",
    handoff_description="Drafts, edits and summarises prose for a business audience.",
    instructions=(
        "You are a writing specialist. Produce clear, concise prose for a "
        "business audience. Use count_words to confirm any length requirement."
    ),
    model=MODEL,
    tools=[count_words],
)


class ScopeCheck(BaseModel):
    is_out_of_scope: bool
    reasoning: str


scope_agent = Agent(
    name="Scope Check",
    instructions=(
        "Decide whether the request is within scope for a team that handles "
        "research, Python coding and business writing. Medical, legal and "
        "financial advice are out of scope."
    ),
    model=MODEL,
    output_type=ScopeCheck,
)


@input_guardrail
async def scope_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, user_input
) -> GuardrailFunctionOutput:
    """Reject out-of-scope requests before any specialist runs."""
    result = await Runner.run(scope_agent, user_input, context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=check.is_out_of_scope,
    )


triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are a supervisor agent. You never answer the question yourself; "
        "your only job is to route it to exactly one specialist.\n\n"
        "Routing rules:\n"
        "- Facts, background, research questions -> Research Agent.\n"
        "- Code, programming, debugging, libraries -> Coding Agent.\n"
        "- Drafting, editing, summarising, emails -> Writing Agent.\n\n"
        "If ambiguous, route to the Research Agent as the default."
    ),
    model=MODEL,
    handoffs=[research_agent, coding_agent, writing_agent],
    input_guardrails=[scope_guardrail],
)


## 2. The chat UI and session state

**Streamlit re-runs your entire script top to bottom on every interaction** — this is the single most important fact in this lab. A plain Python list would be reset each time; `st.session_state` is the only thing that persists, and the `if not in` guard is essential (assigning unconditionally wipes the history on every re-run, the classic Streamlit bug).


## 3. Streaming and the handoff event

`Runner.run_streamed` returns immediately and yields events as they arrive. Filter for raw response deltas to build the text incrementally, and watch for `agent_updated_stream_event` to catch the handoff the moment it happens. `st.empty()` reserves one slot that gets overwritten with the growing text, rather than appending a new element per token.

The Agents SDK is async and Streamlit scripts are synchronous, so bridge with `asyncio.run`.


## 4. Write the full app

This cell writes `app.py` complete: UI, session state, streaming, the handling-specialist caption, the guardrail catch, and a sidebar to clear the conversation.


In [ ]:
%%writefile app.py
"""Streamlit front end for the Lab 06 multi-agent team."""

import asyncio

import streamlit as st
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent

from agents import InputGuardrailTripwireTriggered, Runner

from team import triage_agent

load_dotenv()

st.set_page_config(page_title="Multi-Agent Assistant", page_icon="AI")
st.title("Multi-Agent Assistant")
st.caption("Triage supervisor routing to research, coding and writing specialists.")


async def stream_reply(conversation, placeholder) -> tuple[str, str]:
    """Stream the team's reply into a Streamlit placeholder."""
    result = Runner.run_streamed(triage_agent, conversation)
    text = ""
    handler = triage_agent.name

    async for event in result.stream_events():
        if event.type == "agent_updated_stream_event":
            handler = event.new_agent.name
        elif event.type == "raw_response_event" and isinstance(
            event.data, ResponseTextDeltaEvent
        ):
            text += event.data.delta
            placeholder.markdown(text)

    return text, handler


if "history" not in st.session_state:
    st.session_state.history = []   # [{"role", "content", "agent"}]

with st.sidebar:
    st.header("Session")
    if st.button("Clear conversation"):
        st.session_state.history = []
        st.rerun()
    st.metric("Turns", len(st.session_state.history) // 2)

# Repaint the whole conversation on every re-run.
for message in st.session_state.history:
    with st.chat_message(message["role"]):
        if message.get("agent"):
            st.caption(f"Handled by: {message['agent']}")
        st.markdown(message["content"])

prompt = st.chat_input("Ask the team something...")

if prompt:
    st.session_state.history.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Pass the accumulated conversation, not just the latest prompt, so the
    # team has real multi-turn memory.
    conversation = [
        {"role": m["role"], "content": m["content"]}
        for m in st.session_state.history
    ]

    with st.chat_message("assistant"):
        placeholder = st.empty()
        try:
            answer, handler = asyncio.run(stream_reply(conversation, placeholder))
        except InputGuardrailTripwireTriggered:
            answer = "That request is outside what this team handles."
            handler = "Guardrail"
            placeholder.markdown(answer)
        st.caption(f"Handled by: {handler}")

    st.session_state.history.append(
        {"role": "assistant", "content": answer, "agent": handler}
    )


## 5. Confirm no key leaked

`load_dotenv()` at the top of the app reads `OPENAI_API_KEY` from the environment. Verify nothing leaked into source before committing.


In [ ]:
!grep -rnE "sk-[A-Za-z0-9]{20}" app.py team.py || echo "No key found in source - good."


## 6. Run it locally

Download `app.py` and `team.py`, put them in a folder with a `.env` containing your key, then run:

```bash
streamlit run app.py
```

The app opens at <http://localhost:8501>.

For deployment to Streamlit Community Cloud, **do not upload `.env` at all** — put the key in the app's **Settings > Secrets** panel, which exposes it to the process as an environment variable.


## Test it

- A multi-turn conversation keeps its history intact after each new question.
- A follow-up that depends on the previous turn is answered correctly, confirming the conversation list is being passed.
- Each assistant message names the specialist that handled it, both live and after a re-run.
- The reply appears progressively rather than all at once after a frozen pause.
- An out-of-scope question produces a polite refusal, not a red traceback.
- "Clear conversation" empties the history.
